# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suha-2004/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/suha-2004/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 224 (delta 108), reused 102 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 3.07 MiB | 11.99 MiB/s, done.
Resolving deltas: 100% (108/108), done.


In [2]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
### Finding 1 — Search-performance signals

The research work uses search-performance observations such as impressions, clicks, and ranking-related measures to understand content discoverability.

**Methodology question:** The label or outcome should come from an observed search-performance outcome rather than from a feature derived from the same outcome period. The validation design should keep the information used to construct the outcome separate from the predictors.

### Finding 2 — Content prioritization

The research work uses measurable content and search signals to support decisions about which content should receive attention.

**Methodology question:** A validation design should test whether the observed pattern remains when the model is evaluated on data that was not used during training. A grouped or time-aware split can provide a more honest estimate of generalization.

These questions are intended as constructive checks on whether the validation design supports the strength of the research claim.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

# Final feature set used for this validation audit
feature_cols = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "engagement_rate"
]

# Define the observed outcome
df["is_declining"] = (
    df["impressions_last_30d"]
    < 0.8 * df["impressions_prev_30d"]
).astype(int)

# Grouped client split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]).intersection(
    set(test_df["client_id"])
)

print("Client overlap:", len(overlap))

Train rows: 19166
Test rows: 10834
Train clients: 22
Test clients: 10
Client overlap: 0


In [6]:
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df["is_declining"]
y_test = test_df["is_declining"]

# Convert to numeric
X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_test = X_test.apply(pd.to_numeric, errors="coerce")

# Fill missing values using training medians
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Training feature matrix:", X_train.shape)
print("Testing feature matrix:", X_test.shape)

Training feature matrix: (19166, 7)
Testing feature matrix: (10834, 7)


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)
recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)
f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

print("ROC AUC:", round(roc_auc, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1:", round(f1, 3))

ROC AUC: 0.651
Precision: 0.622
Recall: 0.787
F1: 0.695


### Validation comparison

The comparison distinguishes the earlier evaluation from the grouped client-aware evaluation. The grouped split is treated as the more honest estimate because clients in the test set are not present in the training set.

In [8]:
validation_comparison = pd.DataFrame({
    "Evaluation": [
        "Earlier final evaluation",
        "Grouped client-aware validation"
    ],
    "ROC_AUC": [
        0.625,
        roc_auc
    ],
    "Precision": [
        0.719,
        precision
    ],
    "Recall": [
        0.710,
        recall
    ],
    "F1": [
        0.714,
        f1
    ]
})

validation_comparison

,Evaluation,ROC_AUC,Precision,Recall,F1
0,Earlier final evaluation,0.625000,0.719000,0.710000,0.714000
1,Grouped client-aware validation,0.651392,0.622486,0.786504,0.694949


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
### Leakage audit

The final feature set is checked against fields representing the recent outcome period and target-related fields. Outcome-period fields are kept outside the feature vector so that the model does not directly receive information from the period used to define the outcome.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit

outcome_period_cols = [
    col for col in df.columns
    if "last_30d" in col.lower()
]

used_outcome_cols = set(feature_cols).intersection(
    outcome_period_cols
)

target_like_features = [
    col for col in feature_cols
    if any(
        term in col.lower()
        for term in ["target", "label", "declining"]
    )
]

print("Outcome-period columns:")
print(outcome_period_cols)

print("\nOutcome-period columns used as features:")
print(used_outcome_cols)

print("\nTarget-like columns used as features:")
print(target_like_features)

if len(used_outcome_cols) == 0 and len(target_like_features) == 0:
    print("\nLeakage audit: PASS")
else:
    print("\nLeakage audit: REVIEW")

Outcome-period columns:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']

Outcome-period columns used as features:
set()

Target-like columns used as features:
[]

Leakage audit: PASS


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
### Claim rewrite

**Bold claim:** The model can identify content that will perform better in search.

**Safer claim:** The observed validation results indicate that the model can provide directional decision-support for prioritizing content based on measured historical search and content signals. The results do not establish that the model causes improved search performance or guarantees future ranking outcomes.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("ML-09 SELF-CHECK")
print("================")

print("Dataset shape:", df.shape)

print("Number of features:", len(feature_cols))

print("Grouped validation completed:", True)

print("Client overlap:", len(overlap))

print("Leakage audit:",
      "PASS"
      if len(used_outcome_cols) == 0
      and len(target_like_features) == 0
      else "REVIEW")

print("Claim rewrite completed:", True)

print("Validation comparison rows:",
      len(validation_comparison))

ML-09 SELF-CHECK
Dataset shape: (30000, 45)
Number of features: 7
Grouped validation completed: True
Client overlap: 0
Leakage audit: PASS
Claim rewrite completed: True
Validation comparison rows: 2


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.